# درس 09 - الگوی طراحی فراشناخت


## راه‌اندازی

این نوت‌بوک الگوی طراحی متاکاگنیشن را با استفاده از Microsoft Agent Framework نشان می‌دهد.

**پیش‌نیازها:**
- استقرار Azure OpenAI که از طریق متغیرهای محیطی پیکربندی شده باشد
- Azure CLI با احراز هویت انجام‌شده (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## فراشناخت چیست؟

فراشناخت یعنی **اندیشیدن دربارهٔ اندیشیدن**. در زمینهٔ عامل‌های هوش مصنوعی، یعنی ساخت عامل‌هایی که بتوانند:

- **خودبازاندیشی** دربارهٔ خروجی‌ها و فرآیند استدلال خود
- **خطاها را شناسایی کنند** و به‌جای شکست خاموش، به‌طور مناسب بازیابی شوند
- **ارزیابی کنند** که آیا پاسخ‌هایشان کامل و مفید هستند
- **تطبیق** استراتژی خود وقتی رویکرد اولیه کار نمی‌کند (مثلاً بازگشت به یک سیستم پشتیبان)

یک عامل فراشناختی تنها به سؤال‌ها پاسخ نمی‌دهد — بلکه عملکرد خود را زیر نظر می‌گیرد و به‌صورت پویا تنظیم می‌کند.


## ابزارهای اصلی و پشتیبان

یک الگوی متا-شناختی رایج، **استراتژی پشتیبان** است. عامل ابتدا یک ابزار اصلی را امتحان می‌کند؛ اگر آن شکست بخورد (مثلاً خطای 404)، عامل شکست را تشخیص می‌دهد و به‌طور شفاف به یک ابزار پشتیبان تغییر می‌دهد.

این شبیه سیستم‌های دنیای واقعی است که در آن سرویس‌های اصلی ممکن است در دسترس نباشند و عامل‌ها باید پیش از انتخاب مسیر جایگزین، خود مسئله را تشخیص دهند.

در زیر ما دو ابزار جستجوی پرواز را تعریف می‌کنیم:
- **Primary** — شامل پاریس، توکیو و بارسلونا است
- **Backup** — شامل برلین، سیدنی و نیویورک است


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## عامل خودبازتابی با بازیابی خطا

عامل زیر دستور داده شده است که ابتدا سامانه پروازی اصلی را امتحان کند، خطاها را شناسایی کند، و به‌صورت شفاف به سامانه پشتیبان بازگردد. پس از هر پاسخ، به‌طور خلاصه خودارزیابی می‌کند که آیا به‌طور کامل به سؤال کاربر پاسخ داده است یا خیر.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## الگوی خودارزیابی

جنبهٔ دیگری از فراشناخت، **خودارزیابی** است: یک عامل جدا (یا همان عامل در یک بازبینی دوم) پاسخ را از نظر کامل بودن، دقت، و مفید بودن بررسی می‌کند.

در ادامه یک عامل `ResponseEvaluator` ایجاد می‌کنیم که پاسخ‌های travel-agent را بر اساس سه بعد امتیازدهی می‌کند.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## خلاصه

در این درس یاد گرفتید چگونه با استفاده از Microsoft Agent Framework، **عاملان فراشناختی** بسازید:

- **خوداندیشی**: عاملانی که فرایند استدلال خود را زیر نظر دارند و با شفافیت توضیح می‌دهند چه اتفاقی افتاده است.
- **بازیابی خطا با جایگزین‌ها**: الگویی ابزار اصلی + پشتیبان که طی آن عامل شکست‌ها (مثلاً خطای 404) را تشخیص می‌دهد و به‌طور خودکار منبع جایگزین را امتحان می‌کند.
- **خودارزیابی**: یک عامل ارزیاب جداگانه که پاسخ‌ها را از نظر کامل بودن، دقت و مفید بودن نمره می‌دهد.

این الگوها عاملان را مقاوم‌تر، شفاف‌تر و قابل‌اعتمادتر می‌کنند — ویژگی‌هایی حیاتی برای استقرار در محیط تولید.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
سلب مسئولیت:
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما برای دقت تلاش می‌کنیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است حاوی خطاها یا نادرستی‌هایی باشند. نسخهٔ اصلی سند به زبان مبدا باید به‌عنوان مرجع معتبر در نظر گرفته شود. برای اطلاعات حساس یا حیاتی، ترجمهٔ حرفه‌ای انسانی توصیه می‌شود. ما در قبال هرگونه سوءتفاهم یا تفسیر نادرست ناشی از استفاده از این ترجمه مسئول نیستیم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
